# TASK 1 — Iris Flower Classification

**Objective:** Classify Iris flowers as Setosa, Versicolor, or Virginica using physical measurements.

**Tech stack:** Python, pandas, scikit-learn, matplotlib, seaborn.

This notebook completes the required EDA, visualisation, feature-selection discussion, two classifiers, evaluation, and best-model selection.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

df.head()

## 1. Data inspection and EDA

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nNull values:")
print(df.isnull().sum())

print("\nDescriptive statistics:")
display(df.describe())

print("\nClass distribution:")
display(df["species"].value_counts())

## 2. Pairplot — feature distributions by species

In [ ]:
sns.pairplot(df, hue="species", diag_kind="hist")
plt.suptitle("Iris Feature Relationships by Species", y=1.02)
plt.show()

## 3. Box plots for each feature

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = iris.feature_names

for ax, feature in zip(axes.ravel(), features):
    sns.boxplot(data=df, x="species", y=feature, ax=ax)
    ax.set_title(f"{feature} by Species")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 4. Feature-selection discussion

The visualisations show that **petal length** and **petal width** are generally the most discriminative features: Setosa is strongly separated from the other two species, while Versicolor and Virginica also show useful separation. Sepal measurements provide information but usually have more overlap.

We retain all four features for the baseline models because the Iris dataset is small, all measurements are meaningful, and the model comparison can determine whether the additional features improve performance.

In [ ]:
X = df[iris.feature_names]
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 5. Train two classifiers

We compare Logistic Regression and K-Nearest Neighbours. Standardisation is included in both pipelines so that measurements on different scales do not disproportionately affect the models.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "K-Nearest Neighbours": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    results.append({"Model": name, "Accuracy": acc})

    print("=" * 70)
    print(name)
    print("Accuracy:", round(acc, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, pred))

    cm = confusion_matrix(y_test, pred, labels=iris.target_names)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=iris.target_names,
                yticklabels=iris.target_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df)

## 6. Best-performing model

The best model is selected using **test-set accuracy**, while the classification report confirms precision, recall, and F1-score for each class. If the two models tie, prefer the simpler model and state the tie explicitly.

The cell below automatically declares the best-performing model based on the measured result rather than assuming a result in advance.

In [ ]:
best_row = results_df.iloc[0]
best_model_name = best_row["Model"]
best_accuracy = best_row["Accuracy"]

print(f"Best-performing model: {best_model_name}")
print(f"Test accuracy: {best_accuracy:.2%}")
print("Justification: it achieved the highest test accuracy among the two evaluated classifiers.")

## Conclusion

The notebook completed dataset inspection, EDA, pairplot, box plots, feature-selection discussion, an 80/20 stratified split, two classifiers, accuracy, confusion matrices, precision/recall/F1 evaluation, and automatic best-model selection.